🎲 Catan Winner Prediction
==========================
Takes the Kaggle Catan dataset (lumins/settlers-of-catan-games), hides everything except each player's two starting settlements
(one number per settlement; the CSV has no tile types), and asks an LLM which player won.

Setup: add your Gemini key to Colab Secrets (🔑 on the left) as `GOOGLE_API_KEY` with Notebook access on.

In [ ]:
!pip install -q google-genai kagglehub

import re, glob, time
import pandas as pd
import kagglehub
from google import genai
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-3.1-flash-lite"

path = kagglehub.dataset_download("lumins/settlers-of-catan-games")  # original dataset, includes tile types
df = pd.read_csv(glob.glob(f"{path}/**/*.csv", recursive=True)[0])
df[["gameNum", "player", "settlement1", "settlement2"]].head()

In [ ]:
# This dataset only has one number per starting settlement (no tile types), so that is all we can pass on.
games, winners = {}, {}
for g, grp in df.groupby("gameNum"):
    top = grp[grp.points == grp.points.max()]
    if len(top) != 1:
        continue  # skip ties
    winners[g] = int(top.player.iloc[0])
    games[g] = "\n".join(
        f"Player {int(r.player)}: settlement 1 = {r.settlement1} | settlement 2 = {r.settlement2}"
        for _, r in grp.sort_values("player").iterrows()
    )

print(f"{len(games)} games ready. Example of what the LLM sees:\n\n{games[next(iter(games))]}")

In [ ]:
PROMPT = """You are an expert Settlers of Catan player.
Below are the two starting settlements of each player in a finished 4-player game. Each settlement is given
as a single number from the dataset. Tile types are not available, and the exact meaning of the number is not
documented; it is probably a measure of the settlement's dice-probability strength (higher is likely better).

{game}

One of these players won. Which one? Reply with only the player number."""

errors = set()

def predict(g):
    for attempt in range(6):
        try:
            reply = client.models.generate_content(model=MODEL, contents=PROMPT.format(game=games[g])).text or ""
            m = re.search(r"[1-4]", reply)
            if m:
                return g, int(m.group())
            errors.add(f"no player number in reply: {reply[:80]!r}")
        except Exception as e:
            errors.add(str(e)[:150])
        time.sleep(5 * 2 ** attempt if attempt < 3 else 30)
    return g, None

def run(todo, workers):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(predict, g) for g in todo]
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"Predicting ({workers} at a time)"):
            g, p = f.result()
            if p is not None:
                preds[g] = p

preds = {}
run(list(games), workers=3)
missing = [g for g in games if g not in preds]
if missing:
    print(f"{len(missing)} games had no answer, retrying them one at a time...")
    run(missing, workers=1)

if errors:
    print("\nProblems seen (first few):")
    for e in list(errors)[:5]:
        print(" -", e)

In [ ]:
results = pd.DataFrame({"game": list(games), "predicted": [preds.get(g) for g in games], "actual": [winners[g] for g in games]})
results["correct"] = results.predicted == results.actual

display(results)
answered = results.predicted.notna().sum()
print(f"\nGames with an answer: {answered} / {len(results)}")
print(f"Correct: {results.correct.sum()} / {len(results)}  =  {results.correct.mean():.1%}  (all games)")
if answered:
    print(f"Correct: {results.correct.sum()} / {answered}  =  {results.correct.sum() / answered:.1%}  (answered games only)")
print("(random guessing would get about 25%)")